In [0]:
SELECT COUNT(*) FROM bridge_monitoring.01_bronze.bridge_temperature

In [0]:
SELECT COUNT(*) FROM bridge_monitoring.02_silver.bridge_temperature;

In [0]:
SELECT * 
FROM bridge_monitoring.03_gold.bridge_metrics
ORDER BY window_start, window_end, bridge_id;

In [0]:
WITH temp_agg AS (
  SELECT
    bridge_id,
    bridge_name,
    country,
    location,
    window.start AS window_start,
    window.end   AS window_end,
    AVG(temperature) AS avg_temperature
  FROM bridge_monitoring.`02_silver`.bridge_temperature
  GROUP BY window(event_time, '10 minutes'),
           bridge_id, bridge_name, country, location
),
vib_agg AS (
  SELECT
    bridge_id,
    window.start AS window_start,
    window.end   AS window_end,
    MAX(vibration) AS max_vibration
  FROM bridge_monitoring.`02_silver`.bridge_vibration
  GROUP BY window(event_time, '10 minutes'), bridge_id
),
tilt_agg AS (
  SELECT
    bridge_id,
    window.start AS window_start,
    window.end   AS window_end,
    MAX(tilt_angle) AS max_tilt_angle
  FROM bridge_monitoring.`02_silver`.bridge_tilt
  GROUP BY window(event_time, '10 minutes'), bridge_id
)
SELECT
  t.bridge_id,
  t.bridge_name,
  t.country,
  t.location,
  t.window_start,
  t.window_end,
  ROUND(t.avg_temperature, 2) AS avg_temperature,
  v.max_vibration,
  w.max_tilt_angle
FROM temp_agg t
JOIN vib_agg v
  ON  t.bridge_id    = v.bridge_id
  AND t.window_start = v.window_start
  AND t.window_end   = v.window_end
JOIN tilt_agg w
  ON  t.bridge_id    = w.bridge_id
  AND t.window_start = w.window_start
  AND t.window_end   = w.window_end;